# NFL game predictions

Build a model that estimates who will win and how many points each team will score. Compare the scoring-and-rest baseline with EPA (expected points added) and Elo team ratings. The last section produces a table of upcoming 2026 matchups you can use in a website.

**Run this notebook top to bottom in VS Code or Jupyter.** If you're missing packages, run `%pip install pandas scikit-learn pyarrow certifi` in a notebook cell and restart the kernel if prompted.

**Data source** · [nflverse schedules and results](https://github.com/nflverse/nfldata/blob/master/data/games.csv) · [Column definitions](https://nflreadr.nflverse.com/articles/dictionary_schedules.html)

## 1. Load the schedule and results

By default, the notebook uses `games.csv` in the notebook's working folder if it exists. This works around the SSL error you saw in VS Code. [Download the raw CSV here](https://raw.githubusercontent.com/nflverse/nfldata/master/data/games.csv) and save it as `games.csv`. Refresh that file each week, or set `USE_LOCAL_IF_AVAILABLE = False` to read directly from nflverse.

In [1]:
from pathlib import Path
from urllib.error import URLError

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_URL = "https://raw.githubusercontent.com/nflverse/nfldata/master/data/games.csv"
LOCAL_CSV = Path("games.csv")
USE_LOCAL_IF_AVAILABLE = True
START_SEASON = 2015
VALIDATION_SEASON = 2024
TEST_SEASON = 2025
PREDICTION_SEASON = 2026
LOOKBACK = 5

columns = [
    "game_id", "season", "game_type", "week", "gameday", "gametime",
    "location", "home_team", "away_team", "home_score", "away_score",
]
source = LOCAL_CSV if USE_LOCAL_IF_AVAILABLE and LOCAL_CSV.exists() else DATA_URL

try:
    schedule = pd.read_csv(source, usecols=columns, parse_dates=["gameday"])
except URLError as error:
    raise RuntimeError(
        f"Could not download the schedule. Save games.csv in {Path.cwd()} "
        "and set USE_LOCAL_IF_AVAILABLE = True, then rerun this cell."
    ) from error

schedule = (
    schedule.loc[schedule["season"].ge(START_SEASON - 1)]
    .sort_values(["gameday", "gametime", "game_id"])
    .reset_index(drop=True)
)
assert schedule["game_id"].is_unique, "Expected one row per game."

# The schedule lists kickoff times in Eastern time, including international games.
kickoff = pd.to_datetime(
    schedule["gameday"].dt.strftime("%Y-%m-%d") + " " + schedule["gametime"],
    format="%Y-%m-%d %H:%M",
    errors="coerce",
)
schedule["kickoff_et"] = kickoff.dt.tz_localize("America/New_York")
schedule["has_result"] = schedule[["home_score", "away_score"]].notna().all(axis=1)
schedule["point_margin"] = schedule["home_score"] - schedule["away_score"]

print(f"Loaded {len(schedule):,} games from {source}")
print(f"Working folder: {Path.cwd()}")
display(
    schedule.groupby("season").agg(
        scheduled_games=("game_id", "size"), completed_games=("has_result", "sum")
    ).tail(6)
)

Loaded 3,567 games from games.csv
Working folder: /Users/ogans/Desktop/NFL Game Prediction Project


,scheduled_games,completed_games
season,,
2021,285,285
2022,284,284
2023,285,285
2024,285,285
2025,285,285
2026,272,135


## 2. Turn completed games into team history

The source gives us one row per game. Here we create two rows per completed game, one from each team's perspective. We include playoff results in team history and use regular-season matchups for training, evaluation, and upcoming predictions.

We use history from **earlier game dates**. This applies the same cutoff in historical tests and future predictions and keeps the matchup's own score out of its inputs. The previous five games can span seasons. Rest days are capped at 21 so the offseason is treated as an extended break.

In [2]:
# Treat franchise moves and abbreviation changes as continuous team histories.
franchise_aliases = {
    "OAK": "LV", "STL": "LA", "LAR": "LA", "SD": "LAC", "WSH": "WAS",
}

completed = schedule.loc[schedule["has_result"]].copy()
home_history = completed[["game_id", "gameday", "home_team", "home_score", "away_score"]].copy()
home_history.columns = ["game_id", "gameday", "team", "points_for", "points_against"]
away_history = completed[["game_id", "gameday", "away_team", "away_score", "home_score"]].copy()
away_history.columns = home_history.columns

team_history = pd.concat([home_history, away_history], ignore_index=True)
team_history["team"] = team_history["team"].replace(franchise_aliases)
team_history = team_history.sort_values("gameday")
history_by_team = dict(tuple(team_history.groupby("team")))

display(team_history.tail())

,game_id,gameday,team,points_for,points_against
3419,2026_09_DAL_IND,2026-11-08,IND,23.0,27.0
3418,2026_09_CLE_NO,2026-11-08,NO,23.0,22.0
3417,2026_09_CIN_ATL,2026-11-08,ATL,17.0,34.0
6858,2026_09_GB_NE,2026-11-08,GB,45.0,27.0
6859,2026_09_TB_CHI,2026-11-08,TB,24.0,27.0


## 3. Build the same features for every matchup

Each row gets the home and away teams' average points scored, average points allowed, and days since their last game. We also mark whether the designated home team is actually playing at home. The 2014 games give the earliest 2015 matchups some history.

In [3]:
feature_cols = [
    "home_points_for_avg", "home_points_against_avg", "home_rest_days",
    "away_points_for_avg", "away_points_against_avg", "away_rest_days",
    "home_field",
]

def team_form(team, game_date):
    team = franchise_aliases.get(team, team)
    history = history_by_team.get(team)
    if history is None or pd.isna(game_date):
        return np.nan, np.nan, np.nan

    prior_games = history.loc[history["gameday"].lt(game_date)].tail(LOOKBACK)
    if len(prior_games) < LOOKBACK:
        return np.nan, np.nan, np.nan

    days_since_last_game = (game_date - prior_games["gameday"].iloc[-1]).days
    return (
        prior_games["points_for"].mean(),
        prior_games["points_against"].mean(),
        min(days_since_last_game, 21),
    )

def add_pregame_features(matchups):
    rows = []
    for game in matchups.itertuples(index=False):
        home_form = team_form(game.home_team, game.gameday)
        away_form = team_form(game.away_team, game.gameday)
        home_field = {"Home": 1, "Neutral": 0}.get(game.location, np.nan)
        rows.append((*home_form, *away_form, home_field))

    features = pd.DataFrame(rows, columns=feature_cols, index=matchups.index)
    return matchups.join(features)

historical_games = completed.loc[
    completed["season"].between(START_SEASON, TEST_SEASON)
    & completed["game_type"].eq("REG")
]
history_features = add_pregame_features(historical_games)
model_data = history_features.dropna(subset=feature_cols).copy()
model_data["home_win"] = model_data["point_margin"].gt(0).astype(int)

print(f"Model-ready games: {len(model_data):,} of {len(historical_games):,}")
display(model_data[["game_id", *feature_cols]].head())

Model-ready games: 2,895 of 2,895


,game_id,home_points_for_avg,home_points_against_avg,home_rest_days,away_points_for_avg,away_points_against_avg,away_rest_days,home_field
267,2015_01_PIT_NE,26.8,19.0,21,26.6,20.0,21,1
268,2015_01_CAR_JAX,17.6,21.4,21,22.8,16.0,21,1
269,2015_01_CLE_NYJ,21.2,19.6,21,11.4,23.6,21,1
270,2015_01_GB_CHI,16.6,27.8,21,22.2,18.6,21,1
271,2015_01_IND_BUF,21.0,16.4,21,18.2,24.0,21,1


## 4. Add offensive and defensive EPA per play

EPA measures how much a play changes the offense's expected points. Use the nflverse play-by-play files and keep run and pass plays with EPA, including sacks classified as pass plays. For each team's previous five completed games, divide total offensive EPA by offensive plays and opponent EPA by defensive plays. Higher EPA allowed is worse for the defense. The same five-game cutoff applies to training and upcoming matchups.

The first run downloads and caches one Parquet file per season, roughly 250 MB in total. Historical files are reused. The current season is refreshed when its cache is at least 24 hours old. If the automatic download fails, use the [nflverse play-by-play release page](https://github.com/nflverse/nflverse-data/releases/tag/pbp) and save the named file inside the `pbp_cache` folder shown in the error.

In [4]:
import shutil
import ssl
import subprocess
import time
from urllib.request import urlopen

import certifi

PBP_URL = "https://github.com/nflverse/nflverse-data/releases/download/pbp"
PBP_CACHE = Path("pbp_cache")
PBP_CACHE.mkdir(exist_ok=True)
ssl_context = ssl.create_default_context(cafile=certifi.where())
pbp_columns = ["game_id", "posteam", "defteam", "play_type", "epa"]

def load_pbp_season(season):
    path = PBP_CACHE / f"play_by_play_{season}.parquet"
    exists = path.exists()
    refresh = season == PREDICTION_SEASON and exists and time.time() - path.stat().st_mtime >= 86400

    if not exists or refresh:
        url = f"{PBP_URL}/{path.name}"
        partial = path.with_suffix(".part")
        print(f"Downloading {path.name}...", flush=True)
        try:
            # curl uses the system certificate setup on macOS.
            if shutil.which("curl"):
                subprocess.run(
                    ["curl", "-fLsS", "--retry", "1", "--max-time", "90",
                     url, "-o", str(partial)],
                    check=True,
                )
            else:
                with urlopen(url, context=ssl_context, timeout=60) as response:
                    with partial.open("wb") as output:
                        shutil.copyfileobj(response, output)
            partial.replace(path)
        except (OSError, subprocess.CalledProcessError) as error:
            partial.unlink(missing_ok=True)
            if exists:
                print(f"Refresh failed. Using cached {path.name}.")
            elif season == PREDICTION_SEASON:
                print(f"Could not download {path.name}. Some forecasts may use the baseline.")
                return None
            else:
                raise RuntimeError(f"Download {url} and save it to {path.resolve()}, then rerun.") from error

    return pd.read_parquet(path, columns=pbp_columns)

def summarize_epa(plays):
    plays = plays.loc[plays["play_type"].isin(["pass", "run"])].dropna(
        subset=["game_id", "posteam", "defteam", "epa"]
    )
    offense = (
        plays.groupby(["game_id", "posteam"], observed=True)["epa"]
        .agg(off_epa_sum="sum", off_plays="size")
        .rename_axis(index=["game_id", "team"])
    )
    defense = (
        plays.groupby(["game_id", "defteam"], observed=True)["epa"]
        .agg(def_epa_allowed_sum="sum", def_plays="size")
        .rename_axis(index=["game_id", "team"])
    )
    return offense.join(defense, how="outer").reset_index()

epa_by_season = []
for season in range(START_SEASON - 1, PREDICTION_SEASON + 1):
    plays = load_pbp_season(season)
    if plays is not None:
        epa_by_season.append(summarize_epa(plays))

team_game_epa = pd.concat(epa_by_season, ignore_index=True)
team_game_epa["team"] = team_game_epa["team"].replace(franchise_aliases)
epa_history = team_history.merge(
    team_game_epa, on=["game_id", "team"], how="left", validate="one_to_one"
).sort_values("gameday")
epa_history_by_team = dict(tuple(epa_history.groupby("team")))

print(f"Matched EPA to {epa_history['off_plays'].notna().sum():,} completed team-game records.")

Matched EPA to 6,860 completed team-game records.


In [5]:
epa_cols = [
    "home_off_epa_per_play", "home_def_epa_allowed_per_play",
    "away_off_epa_per_play", "away_def_epa_allowed_per_play",
]
epa_feature_cols = feature_cols + epa_cols
epa_totals = ["off_epa_sum", "off_plays", "def_epa_allowed_sum", "def_plays"]

def team_epa_form(team, game_date):
    team = franchise_aliases.get(team, team)
    history = epa_history_by_team.get(team)
    if history is None or pd.isna(game_date):
        return np.nan, np.nan

    prior = history.loc[history["gameday"].lt(game_date)].tail(LOOKBACK)
    totals = prior[epa_totals].sum(min_count=LOOKBACK)
    return (
        totals["off_epa_sum"] / totals["off_plays"],
        totals["def_epa_allowed_sum"] / totals["def_plays"],
    )

def add_epa_features(matchups):
    rows = [
        (*team_epa_form(game.home_team, game.gameday),
         *team_epa_form(game.away_team, game.gameday))
        for game in matchups.itertuples(index=False)
    ]
    epa_features = pd.DataFrame(rows, columns=epa_cols, index=matchups.index)
    return matchups.join(epa_features)

epa_model_data = add_epa_features(model_data).dropna(subset=epa_cols)
print(f"EPA-ready historical games: {len(epa_model_data):,} of {len(model_data):,}")
display(epa_model_data[["game_id", *epa_cols]].head())

EPA-ready historical games: 2,895 of 2,895


,game_id,home_off_epa_per_play,home_def_epa_allowed_per_play,away_off_epa_per_play,away_def_epa_allowed_per_play
267,2015_01_PIT_NE,0.107919,-0.002000,0.105813,0.019777
268,2015_01_CAR_JAX,-0.130044,-0.037573,-0.020761,-0.184494
269,2015_01_CLE_NYJ,0.003424,-0.014738,-0.243559,-0.048212
270,2015_01_GB_CHI,-0.091961,0.152868,0.011267,-0.111001
271,2015_01_IND_BUF,-0.088912,-0.138727,-0.068838,-0.038387


## 5. Calculate pregame Elo

Elo updates each team's rating based on the result and how strong the opponent was at kickoff. All teams start at 1,500 in 2014. This notebook uses a 20-point update factor, 55 Elo points for home-field advantage, and 25% regression toward 1,500 between seasons. These are fixed starting choices for this experiment.

For each game date, record the home-minus-away rating gap, including home-field advantage where applicable, before applying that day's completed results. A tie counts as half a win. Playoff games also update the ratings, and upcoming matchups use the most recent available ratings.

In [6]:
ELO_START = 1500
ELO_K = 20
ELO_HOME_ADVANTAGE = 55
ELO_OFFSEASON_RETAIN = 0.75
elo_cols = ["pregame_elo_gap"]

def build_elo_features(games):
    ratings = {}
    previous_season = None
    gaps = {}

    for (season, _), day in games.groupby(["season", "gameday"], sort=True):
        if previous_season is not None and season != previous_season:
            ratings = {
                team: ELO_START + ELO_OFFSEASON_RETAIN * (rating - ELO_START)
                for team, rating in ratings.items()
            }
        previous_season = season
        updates = {}

        for game in day.itertuples(index=False):
            home = franchise_aliases.get(game.home_team, game.home_team)
            away = franchise_aliases.get(game.away_team, game.away_team)
            home_rating = ratings.get(home, ELO_START)
            away_rating = ratings.get(away, ELO_START)
            advantage = ELO_HOME_ADVANTAGE if game.location == "Home" else 0
            gap = home_rating - away_rating + advantage
            gaps[game.game_id] = gap

            if game.has_result:
                expected_home = 1 / (1 + 10 ** (-gap / 400))
                actual_home = (np.sign(game.point_margin) + 1) / 2
                change = ELO_K * (actual_home - expected_home)
                updates[home] = updates.get(home, 0) + change
                updates[away] = updates.get(away, 0) - change

        for team, change in updates.items():
            ratings[team] = ratings.get(team, ELO_START) + change

    return pd.Series(gaps, name=elo_cols[0], dtype=float)

pregame_elo = build_elo_features(schedule)

def add_elo_features(matchups):
    return matchups.join(pregame_elo, on="game_id")

elo_model_data = add_elo_features(model_data).dropna(subset=elo_cols)
comparison_data = add_elo_features(epa_model_data).dropna(subset=elo_cols)
print(f"Games available for all four models: {len(comparison_data):,}")
display(elo_model_data[["game_id", *elo_cols]].head())

Games available for all four models: 2,895


,game_id,pregame_elo_gap
267,2015_01_PIT_NE,95.934877
268,2015_01_CAR_JAX,-1.188852
269,2015_01_CLE_NYJ,28.296148
270,2015_01_GB_CHI,-37.722816
271,2015_01_IND_BUF,23.961714


## 6. Compare the four models

Compare the baseline, baseline plus EPA, Elo alone, and baseline plus Elo using the same games for each model. For each validation season from 2022 through 2024, train on earlier seasons. For 2025, train through 2023 to match the original notebook's benchmark. Earlier results within each evaluation season can still update the pregame features for later matchups.

The winner models use logistic regression and skip ties. The score models use ridge regression and include ties. Higher winner accuracy is better. Lower log loss and score MAE are better. Use 2022–2024 to choose the model. Keep 2025 as a reference because its results have already been reviewed.

For the website, consider the two Elo variants for promotion. An Elo variant must beat the baseline in at least two of the three validation seasons and have higher average validation accuracy. If both qualify, choose the higher average accuracy, using lower log loss to break a tie.

In [7]:
score_cols = ["home_score", "away_score"]

def fit_models(games, inputs):
    decisive_games = games.loc[games["point_margin"].ne(0)]
    winner = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    winner.fit(decisive_games[inputs], decisive_games["home_win"])

    scores = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    scores.fit(games[inputs], games[score_cols])
    return winner, scores

def projected_points(model, features):
    return model.predict(features).clip(min=0)

def evaluate(games, winner, scores, inputs):
    decisive = games.loc[games["point_margin"].ne(0)]
    actual_winners = decisive["home_win"]
    home_prob = winner.predict_proba(decisive[inputs])[:, 1]
    predicted_scores = projected_points(scores, games[inputs])

    return {
        "Winner games": len(decisive),
        "Always-home accuracy": actual_winners.mean(),
        "Winner accuracy": accuracy_score(actual_winners, home_prob >= 0.5),
        "Log loss": log_loss(actual_winners, home_prob, labels=[0, 1]),
        "Score MAE": mean_absolute_error(games[score_cols], predicted_scores),
    }


model_features = {
    "Baseline": feature_cols,
    "Baseline + EPA": epa_feature_cols,
    "Elo only": elo_cols,
    "Baseline + Elo": feature_cols + elo_cols,
}
selection_seasons = list(range(VALIDATION_SEASON - 2, VALIDATION_SEASON + 1))

results = {}
for season in [*selection_seasons, TEST_SEASON]:
    train_before = min(season, VALIDATION_SEASON)
    train = comparison_data.loc[comparison_data["season"].lt(train_before)]
    held_out = comparison_data.loc[comparison_data["season"].eq(season)]
    if train.empty or held_out.empty:
        raise ValueError(f"Missing comparable data for the {season} evaluation.")

    for name, inputs in model_features.items():
        winner, scores = fit_models(train, inputs)
        results[(season, name)] = evaluate(held_out, winner, scores, inputs)

metrics = pd.DataFrame.from_dict(results, orient="index")
metrics.index.names = ["Season", "Model"]
metrics["Winner games"] = metrics["Winner games"].astype(int)
display(metrics.round(3))

validation = metrics.loc[selection_seasons]
accuracy_by_season = validation["Winner accuracy"].unstack("Model")
validation_summary = validation.groupby(level="Model")[["Winner accuracy", "Log loss", "Score MAE"]].mean()
validation_summary["Seasons beating baseline"] = accuracy_by_season.gt(
    accuracy_by_season["Baseline"], axis="index"
).sum()
display(validation_summary.round(3))

elo_options = validation_summary.loc[["Elo only", "Baseline + Elo"]]
qualified = elo_options.loc[
    elo_options["Seasons beating baseline"].ge(2)
    & elo_options["Winner accuracy"].gt(validation_summary.loc["Baseline", "Winner accuracy"])
].sort_values(["Winner accuracy", "Log loss"], ascending=[False, True])
selected_model = "Baseline" if qualified.empty else qualified.index[0]
print(f"Selected website model: {selected_model}")

Winner games  Always-home accuracy  Winner accuracy  \
Season Model                                                                 
2022   Baseline                 269                 0.561            0.599   
       Baseline + EPA           269                 0.561            0.610   
       Elo only                 269                 0.561            0.617   
       Baseline + Elo           269                 0.561            0.613   
2023   Baseline                 272                 0.555            0.603   
       Baseline + EPA           272                 0.555            0.603   
       Elo only                 272                 0.555            0.625   
       Baseline + Elo           272                 0.555            0.643   
2024   Baseline                 272                 0.533            0.618   
       Baseline + EPA           272                 0.533            0.610   
       Elo only                 272                 0.533            0.673   
       Baseline + Elo           272                 0.533            0.688   
2025   Baseline                 271                 0.539            0.631   
       Baseline + EPA           271                 0.539            0.620   
       Elo only                 271                 0.539            0.631   
       Baseline + Elo           271                 0.539            0.638   

                       Log loss  Score MAE  
Season Model                                
2022   Baseline           0.660      7.121  
       Baseline + EPA     0.654      7.131  
       Elo only           0.660      7.392  
       Baseline + Elo     0.653      7.154  
2023   Baseline           0.652      7.611  
       Baseline + EPA     0.657      7.621  
       Elo only           0.662      7.651  
       Baseline + Elo     0.649      7.550  
2024   Baseline           0.657      7.442  
       Baseline + EPA     0.652      7.383  
       Elo only           0.617      7.431  
       Baseline + Elo     0.621      7.329  
2025   Baseline           0.622      7.431  
       Baseline + EPA     0.624      7.438  
       Elo only           0.650      7.607  
       Baseline + Elo     0.627      7.408

,Winner accuracy,Log loss,Score MAE,Seasons beating baseline
Model,,,,
Baseline,0.606,0.656,7.391,0
Baseline + EPA,0.608,0.654,7.378,1
Baseline + Elo,0.648,0.641,7.344,3
Elo only,0.638,0.647,7.491,3


Selected website model: Baseline + Elo


## 7. Predict the upcoming 2026 games

The website forecast defaults to the model selected on 2022–2024 validation. You can set `PREDICTION_MODEL` to any of the four model names to try it. The chosen winner and score models refit through 2025. If a matchup lacks inputs for the chosen model, it uses the baseline when its inputs are available. The `model_used` column identifies the model for each matchup.

Refresh the schedule and rerun as new results arrive. Later weeks use games played so far, so their forecasts change as the season progresses. Win probabilities and projected points come from separate models and can favor different teams.

In [8]:
PREDICTION_MODEL = selected_model
as_of = pd.Timestamp.now(tz="America/New_York")
upcoming = schedule.loc[
    schedule["season"].eq(PREDICTION_SEASON)
    & schedule["game_type"].eq("REG")
    & ~schedule["has_result"]
    & schedule["kickoff_et"].gt(as_of)
]
ready = add_elo_features(add_epa_features(add_pregame_features(upcoming)))

output_cols = ["game_id", "season", "week", "kickoff_et", "away_team", "home_team"]

def predict_matchups(matchups, training_games, inputs, model_name):
    output = matchups[output_cols].copy()
    if output.empty:
        return output

    winner, scores = fit_models(training_games, inputs)
    features = matchups[inputs]
    home_prob = winner.predict_proba(features)[:, 1]
    points = projected_points(scores, features)
    output["home_win_probability"] = home_prob
    output["away_win_probability"] = 1 - home_prob
    output["model_pick"] = np.where(home_prob >= 0.5, matchups["home_team"], matchups["away_team"])
    output["projected_home_score"] = np.rint(points[:, 0]).astype(int)
    output["projected_away_score"] = np.rint(points[:, 1]).astype(int)
    output["prediction_time_et"] = as_of
    output["model_used"] = model_name
    return output

training_data = {
    "Baseline": model_data,
    "Baseline + EPA": epa_model_data,
    "Elo only": elo_model_data,
    "Baseline + Elo": elo_model_data,
}
chosen_inputs = model_features[PREDICTION_MODEL]
has_chosen_inputs = ready[chosen_inputs].notna().all(axis=1)
has_baseline_inputs = ready[feature_cols].notna().all(axis=1)
predictions = pd.concat([
    predict_matchups(
        ready.loc[has_chosen_inputs], training_data[PREDICTION_MODEL], chosen_inputs, PREDICTION_MODEL
    ),
    predict_matchups(
        ready.loc[~has_chosen_inputs & has_baseline_inputs], model_data, feature_cols, "Baseline"
    ),
]).sort_values("kickoff_et")

if not predictions.empty:
    projected_score_cols = ["projected_home_score", "projected_away_score"]
    predictions[projected_score_cols] = predictions[projected_score_cols].astype(int)
    next_week = predictions["week"].min()
    next_games = predictions.loc[predictions["week"].eq(next_week)]
    print(f"Showing Week {next_week}. Scored {len(predictions)} future games in total.")
    print(predictions["model_used"].value_counts().to_string())
    display(next_games.round({"home_win_probability": 3, "away_win_probability": 3}))
elif upcoming.empty:
    print(f"No future regular-season kickoffs found for {PREDICTION_SEASON}. Refresh the schedule if needed.")

if len(predictions) < len(upcoming):
    print(f"Skipped {len(upcoming) - len(predictions)} matchups with incomplete features.")

Showing Week 9. Scored 137 future games in total.
model_used
Baseline + Elo    137


,game_id,season,week,kickoff_et,away_team,home_team,home_win_probability,away_win_probability,model_pick,projected_home_score,projected_away_score,prediction_time_et,model_used
3430,2026_09_BUF_MIN,2026,9,2026-11-09 20:15:00-05:00,BUF,MIN,0.466,0.534,BUF,22,23,2026-11-09 19:01:09.490981-05:00,Baseline + Elo


## 8. Export the predictions for the website

This cell writes the upcoming-game table with the same score and probability columns plus `model_used`. Keep dated copies before kickoff if you want to grade the original picks by `game_id`.

In [10]:
if not predictions.empty:
    output_path = Path(f"upcoming_nfl_predictions_{PREDICTION_SEASON}.csv")
    predictions.to_csv(output_path, index=False)
    print(f"Saved {len(predictions)} predictions to {output_path.resolve()}")
else:
    print("No predictions to export.")

Saved 137 predictions to /Users/ogans/Desktop/NFL Game Prediction Project/upcoming_nfl_predictions_2026.csv
